# Which Side Wins, Red or Blue?

**Name(s)**: Milo Palmquist, and Shriya Pattapu

**Website Link**: https://spattapu1.github.io/LoL-Statistical-Analysis/

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

#from dsc80_utils import * # Feel free to uncomment and use this.
import seaborn as sns
import plotly.graph_objs as go
from scipy import stats



## Step 1: Introduction

#### Posited Questions: 

- **Are you more likely perform better if you are on the blue side or red side?**
- Does having a champion with a high ban rate on your increase the likelihood of winning?
- Does bot draw firstblood more than other positions? 

 *Bolded question is the one we chose. 

## Step 2: Data Cleaning and Exploratory Data Analysis

### Data Cleaning

In [2]:
# Load the original data
df = pd.read_csv("main.csv", low_memory=False)

In [5]:
# Only keep the rows where the datacompleteness is 'complete'
complete = df[df['datacompleteness'] == 'complete']

# Convert the gamelength column to a datetime object for readability
complete['gamelength'] = pd.to_datetime(complete['gamelength'], unit= 's').dt.time

# Only keep the columns that are relevant to the analysis, and reset the index
complete_teams = complete.loc[complete['position'] == 'team']
teams = complete_teams[['gameid','league','side','teamname', 'gamelength', 'result', 'teamkills', 'earnedgold', 'damagetochampions', 'xpat25', 'csat25', 'dragons', 'barons']].reset_index(drop=True)
teams.head()

/var/folders/9q/rmwtnl614cz0kwkv_mbv5h780000gn/T/ipykernel_12769/691252172.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  complete['gamelength'] = pd.to_datetime(complete['gamelength'], unit= 's').dt.time


,gameid,league,side,teamname,gamelength,result,teamkills,earnedgold,damagetochampions,xpat25,csat25,dragons,barons
0,ESPORTSTMNT01_2690210,LCKC,Blue,BRION Challengers,00:28:33,0,9,28222.0,56560.0,45960.0,767.0,1.0,0.0
1,ESPORTSTMNT01_2690210,LCKC,Red,Nongshim Esports Academy,00:28:33,1,19,33769.0,79912.0,49931.0,864.0,3.0,0.0
2,ESPORTSTMNT01_2690219,LCKC,Blue,T1 Esports Academy,00:35:14,0,3,34688.0,59579.0,49409.0,895.0,1.0,0.0
3,ESPORTSTMNT01_2690219,LCKC,Red,Liiv SANDBOX Youth,00:35:14,1,16,48063.0,74855.0,57155.0,928.0,4.0,2.0
4,ESPORTSTMNT01_2690227,LCKC,Blue,KT Rolster Challengers,00:32:52,1,14,41372.0,67376.0,52441.0,912.0,4.0,1.0


### Univariate Analysis 

In [6]:
# Distribution of Team Kills per Team
fig = px.histogram(teams, x='teamkills')
fig.update_layout(
    title='Distribution of Team Kills per Team',
    xaxis_title='Number of Team Kills',
    yaxis_title='Count'
)

In [7]:
# Distribution of Earned Gold per Team
fig = px.histogram(teams, x='earnedgold')
fig.update_layout(
    title='Distribution of Earned Gold per Team',
    xaxis_title='Amount of Earned Gold',
    yaxis_title= 'Count'
)

### Bivariate Analysis

In [8]:
# Distribution of Total Kills per Team by Side
fig = go.Figure()
fig.add_trace(go.Histogram(x=teams[teams['side'] == 'Red']['teamkills'], name='Red',marker_color='red'))
fig.add_trace(go.Histogram(x=teams[teams['side'] == 'Blue']['teamkills'], name='Blue',marker_color='blue'))
fig.update_layout(barmode='overlay')
fig.update_traces(opacity=0.50)
fig.update_layout(yaxis_title="Count")
fig.update_layout(xaxis_title="Total Kills")
fig.update_layout(title="Total Kills by Side")
fig.show()

In [9]:
# Distribution of Earned Gold per Team by Side
fig = go.Figure()
fig.add_trace(go.Histogram(x=teams[teams['side'] == 'Red']['earnedgold'], name='Red',marker_color='red'))
fig.add_trace(go.Histogram(x=teams[teams['side'] == 'Blue']['earnedgold'], name='Blue',marker_color='blue'))
fig.update_layout(barmode='overlay')
fig.update_traces(opacity=0.50)
fig.update_layout(yaxis_title="Count")
fig.update_layout(xaxis_title="Total Earned Gold")
fig.update_layout(title="Total Earned Gold by Side")
fig.show()

### Intresting Aggregates

In [10]:
# Grouping the data by league and side
teams.groupby(['league', 'side'])[['result', 'teamkills','earnedgold', 'damagetochampions', 'xpat25', 'csat25', 'dragons', 'barons']].mean()

result  teamkills    earnedgold  damagetochampions  \
league side                                                         
CBLOL  Blue  0.539095  14.728395  37677.148148       67642.057613   
       Red   0.460905  13.761317  36747.370370       66220.884774   
CBLOLA Blue  0.546296  14.546296  37296.722222       67311.032407   
       Red   0.453704  14.611111  37123.902778       67086.601852   
CDF    Blue  0.473684  18.105263  35576.473684       77729.328947   
...               ...        ...           ...                ...   
VCS    Red   0.504615  15.941538  36289.778462       66313.187692   
VL     Blue  0.464706  14.811765  34881.805882       63457.776471   
       Red   0.535294  15.258824  35772.411765       65774.694118   
WLDs   Blue  0.531915  12.865248  36133.453901       68238.085106   
       Red   0.468085  12.992908  35839.078014       66783.120567   

                   xpat25      csat25   dragons    barons  
league side                                                
CBLOL  Blue  51564.808511  823.412766  2.176955  0.736626  
       Red   51371.280851  829.910638  2.444444  0.679012  
CBLOLA Blue  51236.606635  819.710900  2.148148  0.740741  
       Red   51441.601896  822.909953  2.398148  0.699074  
CDF    Blue  51760.205882  776.397059  1.565789  0.578947  
...                   ...         ...       ...       ...  
VCS    Red   53527.910596  846.225166  2.350769  0.704615  
VL     Blue  51285.057692  788.666667  2.194118  0.558824  
       Red   52088.250000  805.307692  2.347059  0.694118  
WLDs   Blue  51742.649254  830.432836  1.936170  0.680851  
       Red   51540.044776  826.776119  2.283688  0.673759  

[102 rows x 8 columns]

In [11]:
# Groupby the data by gameid and side
teams.groupby(['gameid', 'side'])[['result', 'teamkills','earnedgold', 'damagetochampions', 'xpat25', 'csat25', 'dragons', 'barons']].mean()

result  teamkills  earnedgold  damagetochampions  \
gameid                side                                                     
ESPORTSTMNT01_2690210 Blue     0.0        9.0     28222.0            56560.0   
                      Red      1.0       19.0     33769.0            79912.0   
ESPORTSTMNT01_2690219 Blue     0.0        3.0     34688.0            59579.0   
                      Red      1.0       16.0     48063.0            74855.0   
ESPORTSTMNT01_2690227 Blue     1.0       14.0     41372.0            67376.0   
...                            ...        ...         ...                ...   
NA1_4493591166        Red      0.0       15.0     28631.0            62069.0   
NA1_4493652706        Blue     1.0       18.0     31221.0            58565.0   
                      Red      0.0       11.0     22602.0            47157.0   
NA1_4493722220        Blue     0.0       13.0     34657.0            73865.0   
                      Red      1.0       28.0     44015.0            89429.0   

                             xpat25  csat25  dragons  barons  
gameid                side                                    
ESPORTSTMNT01_2690210 Blue  45960.0   767.0      1.0     0.0  
                      Red   49931.0   864.0      3.0     0.0  
ESPORTSTMNT01_2690219 Blue  49409.0   895.0      1.0     0.0  
                      Red   57155.0   928.0      4.0     2.0  
ESPORTSTMNT01_2690227 Blue  52441.0   912.0      4.0     1.0  
...                             ...     ...      ...     ...  
NA1_4493591166        Red   54649.0   732.0      1.0     0.0  
NA1_4493652706        Blue      NaN     NaN      1.0     1.0  
                      Red       NaN     NaN      1.0     0.0  
NA1_4493722220        Blue  51930.0   800.0      0.0     0.0  
                      Red   54461.0   760.0      3.0     2.0  

[21312 rows x 8 columns]

In [12]:
# Groupby the data by side
# Add to final report.
teams.groupby('side')[['result', 'teamkills','earnedgold', 'damagetochampions', 'xpat25', 'csat25', 'dragons', 'barons']].mean()


,result,teamkills,earnedgold,damagetochampions,xpat25,csat25,dragons,barons
side,,,,,,,,
Blue,0.523086,14.873592,36566.605950,67612.102665,51893.280765,820.957645,2.140015,0.669482
Red,0.476914,14.322166,35947.699418,66651.462087,51831.290077,822.529188,2.351351,0.681588


## Step 3: Assessment of Missingness

We do not think there are any columns in the dataset that are NMAR. We believe that all of the missing data is either missing by design or MAR. Because the original dataset includes data for both invidual players and the teams they are on, columns that only contain aggregate team data are empty for rows containing player data, and columns that only contain individual player data are empty for rows containing team data. There are also null values in the columns containing ban information for each team if they banned less than five champions, but these cells are intentionally left empty because there is no data to enter. Additionally, since many columns in the dataset are related to game statistics at certain times in the game, there will be missing data in those columns if a game ends before that time. In this case, the missingness in this column is dependent on the game length, which is also included in the dataset. There are also rows missing data in all of these time columns, but they all rows containing data from League of Legends Pro League (LPL) games, and these rows also have the value 'partial' in the datacompletenesscolumn, therefore this data is also MAR.


In [13]:
xpat25_notna = teams[teams['xpat25'].isna() == False]
xpat25_notna.head()

,gameid,league,side,teamname,gamelength,result,teamkills,earnedgold,damagetochampions,xpat25,csat25,dragons,barons
0,ESPORTSTMNT01_2690210,LCKC,Blue,BRION Challengers,00:28:33,0,9,28222.0,56560.0,45960.0,767.0,1.0,0.0
1,ESPORTSTMNT01_2690210,LCKC,Red,Nongshim Esports Academy,00:28:33,1,19,33769.0,79912.0,49931.0,864.0,3.0,0.0
2,ESPORTSTMNT01_2690219,LCKC,Blue,T1 Esports Academy,00:35:14,0,3,34688.0,59579.0,49409.0,895.0,1.0,0.0
3,ESPORTSTMNT01_2690219,LCKC,Red,Liiv SANDBOX Youth,00:35:14,1,16,48063.0,74855.0,57155.0,928.0,4.0,2.0
4,ESPORTSTMNT01_2690227,LCKC,Blue,KT Rolster Challengers,00:32:52,1,14,41372.0,67376.0,52441.0,912.0,4.0,1.0


In [ ]:
xpat25_na = teams[teams['xpat25'].isna() == True]
xpat25_na.head()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=xpat25_notna['gamelength'], name='Not Missing',marker_color='red', nbinsx=int(xpat25_notna['gamelength'].max()-xpat25_notna['gamelength'].min())))
fig.add_trace(go.Histogram(x=xpat25_na['gamelength'], name='Missing',marker_color='blue', nbinsx=int(xpat25_na['gamelength'].max()-xpat25_na['gamelength'].min())))
fig.update_layout(barmode='overlay')
fig.update_traces(opacity=0.55)
fig.update_layout(yaxis_title="count")
fig.update_layout(xaxis_title="Game Length (minutes)")
fig.update_layout(title="Game Length by Missingness of xpat25")
fig

In [ ]:
observed_ks = stats.ks_2samp(xpat25_notna['gamelength'], xpat25_na['gamelength']).statistic
observed_ks

In [ ]:
shuffled = teams.copy()

ks_stats = []
for _ in range(1000):
    
    shuffled['gamelength'] = np.random.permutation(shuffled['gamelength'])
    
    ks_stat = stats.ks_2samp(shuffled[shuffled['xpat25'].isna() == False]['gamelength'], shuffled[shuffled['xpat25'].isna() == True]['gamelength']).statistic
    ks_stats.append(ks_stat)

ks_stats[:10]

In [ ]:
ks_pval = (ks_stats >= observed_ks).mean()
ks_pval

In [ ]:
side_dist = (
    teams
    .assign(xpat25_missing=teams['xpat25'].isna())
    .pivot_table(index='side', columns='xpat25_missing', aggfunc='size')
)

side_dist.columns = ['xpat25_missing = False', 'xpat25_missing = True']

side_dist = side_dist / side_dist.sum()
side_dist

In [ ]:
side_dist.plot(kind='barh', title='Side by Missingness of xpat25', barmode='group')


In [ ]:
observed_tvd = np.float64(sum(abs(side_dist['xpat25_missing = False'] - side_dist['xpat25_missing = True']))/2)
observed_tvd

In [ ]:
shuffled = teams.copy()

tvds = []
for i in range(1000):

    shuffled['side'] = np.random.permutation(shuffled['side'])

    side_dist = (
        shuffled
        .assign(xpat25_missing=teams['xpat25'].isna())
        .pivot_table(index='side', columns='xpat25_missing', aggfunc='size')
    )

    side_dist.columns = ['xpat25_missing = False', 'xpat25_missing = True']

    side_dist = side_dist / side_dist.sum()
    tvd = sum(abs(side_dist['xpat25_missing = False'] - side_dist['xpat25_missing = True']))/2
    tvds.append(tvd)

tvds[:10]

In [ ]:
tvd_pval = (tvds >= observed_tvd).mean()
tvd_pval

## Step 4: Hypothesis Testing

**Null Hypothesis**: The average team kills for blue and red sides is the same.

**Alternative Hypothesis**: The average team kills for blue side is greater then red side.

**Test-Statistic**: Difference in Means of team kills for blue side and team kills for red side.


In [ ]:
# Create a sampling distribution of the difference in means of team kills between the Blue and Red side
sampling_dist = []
for i in range(1000):
    sample = teams.sample(2500)
    by_side = sample.groupby('side')['teamkills'].mean()
    sampling_dist.append(by_side['Blue'] - by_side['Red'])

sampling_dist = pd.DataFrame(sampling_dist, columns=["Difference in Means of Team Kills (Blue - Red)"])
sampling_dist.head()

In [ ]:
fig = px.histogram(sampling_dist, 'Difference in Means of Team Kills (Blue - Red)')
fig.update_layout(yaxis_title="Count")
fig.update_layout(xaxis_title="Difference in Means of Team Kills (Blue - Red)")
fig.update_layout(title="Sampling Distribution of the Difference in Means of Team Kills (Blue - Red)")
fig.show()

In [ ]:
#observed = teams.groupby('side')['teamkills'].mean().loc['Blue'] - teams.groupby('side')['teamkills'].mean().loc['Red']
p_val = ((sampling_dist['Difference in Means of Team Kills (Blue - Red)']) <= 0).mean()
p_val

## Step 5: Framing a Prediction Problem

Previously, we have found that being on blue side may have a significant affect on team kills. Since statistics for blue and red side are different, are there specific statistics of gameplay like xpat25, csat25, barons, dragons, etc. that are higher as a result of being on Blue or Red side, and can we use these statistics to predict which side the player was on?

To address this question, we can use a binary classification algorithm to predict wether or not they were on blue or red side. Therefore, our prediction problem is: Are we able to predict wether a team was blue or red side in a game based on other in-game statistics. In this model we intended to predict the side of the team based on teamkills and earnedgold. We attempted to do this for our baseline model which resulted in a model that was not accurate, which we will address in the next section. 

But as a result, we created a new iteration of our prediction problem, which is: Based on the difference in in-game statistics can we predict wether the winning side of a game was blue or red?. This allows for our classification model to have more accurate measures of differences between sides as apposed to general in-game statistics that are not in relation to the other side. Thus our response variable is wether the winner was red or blue which is included in the differences between team statistics dataframe. 

In this section, we queried out gamelength less than 25 minutes in order to ensure no missing values in xpat25 and csat25. If a game ended before that time, games won't have xp or cs at 25 minutes, and since we want to be able to make accurate predictions the average game (most of which are greater than 25 minutes, the average game is 30 minutes). 

In order to evaluate our model we will only be using accuracy. The reason we are not using F-1 scores and only using accuracy because our data is balanced with each game having a red or blue side, and therefore do not have to consider false positives and negatives in the evalution of our model. 

The information that we would know at the time of prediction for the final model is the differences between the statistics for teamkills, earnedgold, damagetochampions, xpat25, csat25, dragons, and barons for our final model and the statistics themselves for the basline model. 

In [ ]:
# Games that lasted more than 25 minutes to be used for final model
at_least_25 = complete_teams[complete_teams['gamelength'] > pd.to_datetime('00:25:00', format='%H:%M:%S').time()][['gameid','league','side','teamname','result', 'teamkills', 'earnedgold', 'damagetochampions', 'xpat25', 'csat25', 'dragons', 'barons']].reset_index(drop=True)
at_least_25.head()

In [ ]:
# Create a dictionary of the differences in the means of the features for each game
unique_games = at_least_25['gameid'].unique()

cols = ['teamkills',
       'earnedgold', 'damagetochampions', 'xpat25', 'csat25', 'dragons',
       'barons']

games_dict = {}
for game in unique_games:
    temp = at_least_25[at_least_25['gameid'] == game]
    
    diffs = []
    for col in cols:
        diff = -1 * (temp[col].diff().iloc[1])
        diffs.append(diff)
    
    games_dict[game] = diffs

In [ ]:
# Create a dataframe from the dictionary with differenes and the winner of the game
diffs_df = pd.DataFrame(games_dict).T
diffs_df.columns = cols
winners = teams[teams['result'] == 1][['gameid', 'side']]
diffs_df = diffs_df.merge(winners, how='inner', left_index=True, right_on='gameid')
diffs_df = diffs_df.set_index('gameid')
diffs_df = diffs_df.rename(columns={'side': 'winner'})
diffs_df.head()

## Step 6: Baseline Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import GridSearchCV

In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(teams[['teamkills', 'earnedgold']], teams['side'], random_state=1)
)

dtb = DecisionTreeClassifier(max_depth=4, criterion='entropy')
dtb.fit(X_train, y_train)

plt.figure(figsize=(15, 5))
plot_tree(dtb, feature_names=X_train.columns, class_names=['blue', 'red'], 
          filled=True, fontsize=10, impurity=False);

dtb.score(X_train, y_train), dtb.score(X_test, y_test)

### Base Model w/ Random Forest Classifier based on the in-game statistics can we predict wether the winning side of a game was blue or red

In [ ]:
at_least_25['winner'] = at_least_25.apply(lambda row: row['side'] if row['result'] == 1 else ('Red' if row['side'] == 'Blue' else 'Blue'), axis=1)

In [ ]:
X_raw = at_least_25[['teamkills', 'earnedgold', 'damagetochampions', 'xpat25', 'csat25', 'dragons', 'barons']]
y_raw = at_least_25['winner']

X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, random_state=1)

raw_model = RandomForestClassifier()
raw_model.fit(X_train, y_train)

In [ ]:
raw_model.score(X_train, y_train), raw_model.score(X_test, y_test)

### Base Model w/ Random Forest Classifier based on the difference in teamkills and earned gold can we predict wether the winning side of a game was blue or red

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(diffs_df[['teamkills', 'earnedgold']], diffs_df['winner'], random_state=1)
diffs_model_on_eg_tk = RandomForestClassifier()
diffs_model_on_eg_tk.fit(X_train, y_train)


In [ ]:
# This model is actually really good. Differences make a lot of difference lol. 
diffs_model_on_eg_tk.score(X_train, y_train), diffs_model_on_eg_tk.score(X_test, y_test)

## Step 7: Final Model

### Final Model w/ Decision Tree

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = (
    train_test_split(diffs_df.drop(columns='winner'), diffs_df['winner'], random_state=1)
)

# Fit the decision tree model
dtf = DecisionTreeClassifier(max_depth=4, criterion='entropy')
dtf.fit(X_train, y_train)


# Plot the decision tree
plt.figure(figsize=(30, 20))
plot_tree(dtf, feature_names=X_train.columns, class_names=['blue', 'red'], 
          filled=True, fontsize=10, impurity=False);

In [ ]:
# Calculate the accuracy of the model
dtf.score(X_train, y_train), dtf.score(X_test, y_test)

### Final Model w/ Random Tree Classifier

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = (
    train_test_split(diffs_df.drop(columns=['winner']), diffs_df['winner'], random_state=1)
)
# Fit the random forest model
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

In [ ]:
# Calculate the accuracy of the model
clf.score(X_train, y_train) , clf.score(X_test, y_test)

### Grid Search for Best Hyperparameters for DecisionTreeClassifier

In [ ]:
# Grid search for best hyperparameters
# Update the hyperparameters dictionary to things that actually matter.  
hyperparameters = {
    'max_depth': [2, 3, 4, 5, 7, 10, 13, 15, 18, None], 
    'min_samples_split': [2, 5, 10, 20, 50, 100, 200],
    'criterion': ['gini', 'entropy']
}

len(hyperparameters['max_depth']) * \
len(hyperparameters['min_samples_split']) * \
len(hyperparameters['criterion'])

searcher = GridSearchCV(DecisionTreeClassifier(), hyperparameters, cv=5)
searcher.fit(X_train, y_train)
searcher.best_params_

In [ ]:
# Get the best model
searcher.cv_results_['mean_test_score']
pd.DataFrame(np.vstack([searcher.cv_results_[f'split{i}_test_score'] for i in range(5)]))

In [ ]:
# Fit the best model
final_tree = DecisionTreeClassifier(**searcher.best_params_)
final_tree.fit(X_train, y_train)
final_tree.score(X_train, y_train), final_tree.score(X_test, y_test)

## Step 8: Fairness Analysis

In [ ]:
# TODO